# LeetCode 1871 — Jump Game VII bằng Python

Link đề: https://leetcode.com/problems/jump-game-vii/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming, Sliding Window, Prefix Sum, Breadth-First Search

Skeleton LeetCode:

```python
class Solution:
    def canReach(self, s: str, minJump: int, maxJump: int) -> bool:
        ...
```

Lộ trình của notebook:
1. **BFS/DFS ngây thơ** — từ mỗi ô đã tới được, thử mọi bước nhảy trong `[minJump, maxJump]` (`O(n · (maxJump − minJump))`).
2. **DP "kéo" (pull)** — `ok[j]` đúng nếu có `i ∈ [j − maxJump, j − minJump]` với `ok[i]` đúng; vẫn `O(n · k)` nếu kiểm tra thẳng.
3. **DP + tổng tiền tố** — đếm số ô "tới được" trong cửa sổ bằng hiệu hai tổng tiền tố ⇒ `O(n)`.
4. **DP + biến đếm cửa sổ trượt** — giữ trực tiếp số ô tới được trong cửa sổ, cộng khi vào, trừ khi ra ⇒ `O(n)`, gọn nhất.
5. **BFS + con trỏ "đã quét tới đâu"** — mỗi ô chỉ được xét một lần ⇒ `O(n)`.

## 1. Mục tiêu học

1. Mô hình hóa bài toán "nhảy trên mảng" thành **đồ thị có hướng** (cạnh `i → j` khi `j − i ∈ [minJump, maxJump]` và `s[j] == '0'`) và thấy vì sao duyệt thẳng tốn `O(n · k)` cạnh.
2. Viết **quy hoạch động dạng "kéo"**: trạng thái của ô `j` phụ thuộc một **cửa sổ** các ô phía trước có độ dài cố định.
3. Dùng **tổng tiền tố** hoặc **bộ đếm cửa sổ trượt** để trả lời câu hỏi "trong cửa sổ có ít nhất một ô tới được không?" trong `O(1)`.
4. Cài **BFS với con trỏ "biên đã quét"** để mỗi chỉ số được thêm vào hàng đợi tối đa một lần — kỹ thuật chung cho các bài "nhảy trong khoảng".
5. Kiểm chứng các cách tối ưu bằng BFS vét cạn trên chuỗi ngẫu nhiên, xử lý cẩn thận chỉ số biên (`j − minJump`, `j − maxJump − 1`).

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s` (chỉ số từ 0) và hai số nguyên `minJump`, `maxJump`. Ban đầu bạn đứng ở chỉ số `0` (bảo đảm `s[0] == '0'`). Bạn có thể nhảy từ `i` tới `j` nếu:
- `i + minJump ≤ j ≤ min(i + maxJump, len(s) − 1)`, và
- `s[j] == '0'`.

Trả về `True` nếu có thể tới chỉ số **cuối** `len(s) − 1`, ngược lại `False`.

**Ví dụ 1:** `s = "011010"`, `minJump = 2`, `maxJump = 3` → `True`. Nhảy `0 → 3` rồi `3 → 5`.

**Ví dụ 2:** `s = "01101110"`, `minJump = 2`, `maxJump = 3` → `False`.

**Ràng buộc:**
- `2 ≤ s.length ≤ 10⁵`
- `s[i]` là `'0'` hoặc `'1'`, `s[0] == '0'`
- `1 ≤ minJump ≤ maxJump < s.length`

Lưu ý: nếu ô cuối là `'1'` thì chắc chắn `False` (không bao giờ đứng được lên ô `'1'`).

In [1]:
import random
import time
from collections import deque
from itertools import accumulate
from typing import List

examples = [
    (("011010", 2, 3), True),
    (("01101110", 2, 3), False),
]
for (s, lo, hi), want in examples:
    print(f"s={s!r}, minJump={lo}, maxJump={hi} -> kỳ vọng {want}")

s='011010', minJump=2, maxJump=3 -> kỳ vọng True
s='01101110', minJump=2, maxJump=3 -> kỳ vọng False


## 3. Kiến thức nền

### 3.1 Duyệt chuỗi theo chỉ số

`s[j] == '0'` kiểm tra ô `j` có đứng được không. `range(a, b + 1)` liệt kê các chỉ số trong đoạn đóng `[a, b]`. Khi `a > b`, `range` rỗng — tiện cho các đoạn "bị cắt" ở biên.

In [2]:
s = "011010"
print([j for j in range(len(s)) if s[j] == '0'], "là các ô đứng được")
print(list(range(5, 3)), "(đoạn rỗng khi a > b)")

[0, 3, 5] là các ô đứng được
[] (đoạn rỗng khi a > b)


### 3.2 BFS với `deque`

Hàng đợi FIFO: `append` vào cuối, `popleft` ở đầu. Dùng mảng `seen` để không thêm một ô hai lần. Với bài "có tới được không" (không cần ngắn nhất), DFS bằng stack cũng được — nhưng BFS giúp cài mẹo "biên đã quét" ở Cách 5.

In [3]:
q = deque([0]); seen = {0}
for nxt in [2, 3, 2]:
    if nxt not in seen:
        seen.add(nxt); q.append(nxt)
print(q, seen)

deque([0, 2, 3]) {0, 2, 3}


### 3.3 Tổng tiền tố trên mảng 0/1

Với mảng `ok` gồm 0/1, `pre[k] = ok[0] + … + ok[k−1]`. Số phần tử bằng 1 trong đoạn `[l, r]` là `pre[r + 1] − pre[l]`. Khác 0 ⇔ trong đoạn có ít nhất một ô tới được.

In [4]:
ok = [1, 0, 0, 1, 0, 1]
pre = [0] + list(accumulate(ok))
print("pre =", pre, "| số 1 trong [1, 3] =", pre[4] - pre[1])

pre = [0, 1, 1, 1, 2, 2, 3] | số 1 trong [1, 3] = 1


### 3.4 Cửa sổ trượt có độ dài cố định

Khi `j` tăng 1, cửa sổ `[j − maxJump, j − minJump]` dịch sang phải đúng 1: phần tử `j − minJump` **vào**, phần tử `j − maxJump − 1` **ra**. Giữ một biến đếm `cnt` = tổng các `ok` trong cửa sổ, cập nhật `O(1)` mỗi bước.

In [5]:
arr = [1, 0, 1, 1, 0, 0, 1]
lo, hi = 2, 3
for j in range(len(arr)):
    l, r = j - hi, j - lo
    window = [arr[i] for i in range(max(0, l), r + 1)] if r >= 0 else []
    print(f"j={j}: cửa sổ chỉ số [{l}, {r}] -> {window}")

j=0: cửa sổ chỉ số [-3, -2] -> []
j=1: cửa sổ chỉ số [-2, -1] -> []
j=2: cửa sổ chỉ số [-1, 0] -> [1]
j=3: cửa sổ chỉ số [0, 1] -> [1, 0]
j=4: cửa sổ chỉ số [1, 2] -> [0, 1]
j=5: cửa sổ chỉ số [2, 3] -> [1, 1]
j=6: cửa sổ chỉ số [3, 4] -> [1, 0]


## 4. Ý tưởng nền tảng / trực giác

Làm tay ví dụ 1 `s = "011010"`, `minJump = 2`, `maxJump = 3`. Ghi `ok[j]` = "đứng được ở `j`":

| j | s[j] | các ô có thể nhảy tới j (`j−3 … j−2`) | có ô nào ok? | ok[j] |
|---|---|---|---|---|
| 0 | 0 | — (điểm xuất phát) | — | **1** |
| 1 | 1 | — | — | 0 (ô '1') |
| 2 | 1 | {0} | có | 0 (ô '1') |
| 3 | 0 | {0, 1} | có (0) | **1** |
| 4 | 1 | {1, 2} | — | 0 (ô '1') |
| 5 | 0 | {2, 3} | có (3) | **1** |

`ok[5] = 1` ⇒ `True`.

Ví dụ 2 `s = "01101110"`: ô 3 tới được từ 0; ô 7 cần một ô ok trong `{4, 5}` — cả hai là `'1'` ⇒ `False`.

**Quan sát:** câu hỏi cho mỗi `j` chỉ là "cửa sổ `[j − maxJump, j − minJump]` có chứa ô ok nào không?" — một truy vấn trên **đoạn liên tiếp** dịch dần sang phải.

## 5. Cách 1 — Brute Force / baseline: BFS thử mọi bước nhảy

**Ý tưởng:** BFS từ 0. Với mỗi ô `i` lấy ra, thử mọi `j ∈ [i + minJump, min(i + maxJump, n − 1)]`; nếu `s[j] == '0'` và chưa thăm thì thêm vào hàng đợi. Trả về `True` khi chạm `n − 1`.

In [6]:
def can_reach_bfs_naive(s, minJump, maxJump):
    n = len(s)
    seen = [False] * n
    seen[0] = True
    q = deque([0])
    while q:
        i = q.popleft()
        if i == n - 1:
            return True
        for j in range(i + minJump, min(i + maxJump, n - 1) + 1):
            if s[j] == '0' and not seen[j]:
                seen[j] = True
                q.append(j)
    return False

for (s, lo, hi), want in examples:
    print(can_reach_bfs_naive(s, lo, hi), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False


**Độ phức tạp:** mỗi ô lấy ra duyệt `k = maxJump − minJump + 1` láng giềng ⇒ Time `O(n · k)`, Space `O(n)`. Worst case `s = "000…0"`, `minJump = 1`, `maxJump = n − 1`: khoảng `n²/2 = 5·10⁹` phép.

**Nút thắt:** các khoảng `[i + minJump, i + maxJump]` của các ô `i` liên tiếp **chồng lên nhau gần hết** — ta xét đi xét lại cùng một ô `j` rất nhiều lần (dù `seen` chặn việc thêm lại, vòng lặp vẫn chạy qua).

In [7]:
for n in [2000, 6000]:
    s = "0" * n
    t0 = time.perf_counter(); r = can_reach_bfs_naive(s, 1, n - 1)
    # đường đi tới đích ngay ô 0 -> n-1; để thấy worst case, đặt ô cuối là '1'
    s_bad = "0" * (n - 1) + "1"
    t1 = time.perf_counter(); r2 = can_reach_bfs_naive(s_bad, 1, n - 1)
    print(f"n={n}: kết quả {r} ({t1 - t0:.3f}s) | ô cuối '1': {r2} ({time.perf_counter() - t1:.3f}s)")

n=2000: kết quả True (0.044s) | ô cuối '1': False (0.042s)


n=6000: kết quả True (0.380s) | ô cuối '1': False (0.379s)


### 5.1 Cách 1b — DP "kéo", kiểm tra cửa sổ trực tiếp

`ok[0] = True`; với `j ≥ 1`: `ok[j] = (s[j] == '0') and any(ok[i] for i in [j − maxJump, j − minJump])`. Cùng độ phức tạp `O(n · k)` nhưng là cầu nối sang các cách tối ưu — vì ta chỉ cần **tăng tốc câu hỏi `any(...)` trên cửa sổ**.

In [8]:
def can_reach_dp_naive(s, minJump, maxJump):
    n = len(s)
    ok = [False] * n
    ok[0] = True
    for j in range(1, n):
        if s[j] != '0':
            continue
        l, r = max(0, j - maxJump), j - minJump
        ok[j] = r >= 0 and any(ok[i] for i in range(l, r + 1))
    return ok[-1]

for (s, lo, hi), want in examples:
    print(can_reach_dp_naive(s, lo, hi), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — đổi "đẩy" thành "kéo".** Thay vì từ `i` đẩy sang mọi `j`, ta hỏi ngược: `j` có ô tiền nhiệm hợp lệ nào không? Các tiền nhiệm của `j` nằm trong **một đoạn liên tiếp** `W(j) = [j − maxJump, j − minJump]`.

**Insight 2 — đoạn trượt đều.** `W(j + 1)` là `W(j)` dịch phải 1. Nếu giữ `cnt(j) = #{i ∈ W(j) : ok[i]}` thì:

`cnt(j + 1) = cnt(j) + ok[j + 1 − minJump] − ok[j − maxJump]`.

Mỗi bước `O(1)` ⇒ cả thuật toán `O(n)`.

**Insight 3 (BFS) — không quét lại vùng đã quét.** Khi BFS lấy ô `i` ra, khoảng cần xét là `[i + minJump, i + maxJump]`. Vì các ô được lấy ra theo thứ tự **tăng dần** (BFS trên đồ thị chỉ có cạnh đi tới chỉ số lớn hơn, và hàng đợi luôn tăng), mọi ô `≤ far` (biên phải lớn nhất đã quét) đều đã được xét ⇒ chỉ cần quét từ `max(i + minJump, far + 1)`.

**Invariant chung:** khi xét `j`, mọi `ok[i]` với `i < j` đã chính xác.

**Quy tắc nếu… thì…:**
- Nếu `s[j] == '1'` thì `ok[j] = False` (nhưng vẫn phải cập nhật bộ đếm cửa sổ đúng!).
- Nếu `j < minJump` thì không có tiền nhiệm ⇒ `False` (trừ `j = 0`).
- Nếu `s[-1] == '1'` có thể trả `False` ngay.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP + tổng tiền tố

Từng đoạn:
1. `ok[0] = 1`, `pre[1] = 1` (pre có độ dài `n + 1`).
2. Với `j = 1..n−1`: `l = max(0, j − maxJump)`, `r = j − minJump`. Nếu `s[j] == '0'` và `r ≥ 0` và `pre[r + 1] − pre[l] > 0` thì `ok[j] = 1`.
3. `pre[j + 1] = pre[j] + ok[j]` — cập nhật **sau** khi tính `ok[j]`, vì cửa sổ của `j` chỉ gồm chỉ số `≤ j − minJump < j`.

In [9]:
def can_reach_prefix(s, minJump, maxJump):
    n = len(s)
    ok = [0] * n
    ok[0] = 1
    pre = [0] * (n + 1)
    pre[1] = 1
    for j in range(1, n):
        if s[j] == '0':
            l, r = max(0, j - maxJump), j - minJump
            if r >= 0 and pre[r + 1] - pre[l] > 0:
                ok[j] = 1
        pre[j + 1] = pre[j] + ok[j]
    return ok[-1] == 1

for (s, lo, hi), want in examples:
    print(can_reach_prefix(s, lo, hi), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def canReach(self, s: str, minJump: int, maxJump: int) -> bool:
        n = len(s)
        ok = [0] * n
        ok[0] = 1
        pre = [0] * (n + 1)
        pre[1] = 1
        for j in range(1, n):
            if s[j] == '0':
                l, r = max(0, j - maxJump), j - minJump
                if r >= 0 and pre[r + 1] - pre[l] > 0:
                    ok[j] = 1
            pre[j + 1] = pre[j] + ok[j]
        return ok[-1] == 1
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)` (hai mảng).

### 7.2 Cách 3 — DP + bộ đếm cửa sổ trượt

Từng đoạn:
1. `cnt` = số ô ok trong cửa sổ hiện tại.
2. Tại `j`: nếu `j − minJump ≥ 0` thì `cnt += ok[j − minJump]` (phần tử **vào**). Nếu `j − maxJump − 1 ≥ 0` thì `cnt −= ok[j − maxJump − 1]` (phần tử **ra**).
3. `ok[j] = s[j] == '0' and cnt > 0`.

Không cần mảng `pre`, chỉ một số nguyên.

In [10]:
def can_reach_window(s, minJump, maxJump):
    n = len(s)
    if s[-1] != '0':
        return False
    ok = [False] * n
    ok[0] = True
    cnt = 0
    for j in range(1, n):
        if j - minJump >= 0:
            cnt += ok[j - minJump]
        if j - maxJump - 1 >= 0:
            cnt -= ok[j - maxJump - 1]
        ok[j] = s[j] == '0' and cnt > 0
    return ok[-1]

for (s, lo, hi), want in examples:
    print(can_reach_window(s, lo, hi), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False


Mẹo Python: `cnt += ok[...]` với `ok` là `bool` vẫn đúng vì `True == 1`, `False == 0`.

**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho `ok` (bộ đếm `O(1)`).

### 7.3 Cách 4 — BFS + biên "đã quét"

Từng đoạn:
1. `q = deque([0])`, `far = 0` (chỉ số lớn nhất đã được quét như một ứng viên).
2. Lấy `i`: quét `j` từ `max(i + minJump, far + 1)` tới `min(i + maxJump, n − 1)`; mỗi `j` có `s[j] == '0'` thì đưa vào hàng đợi (chắc chắn chưa thăm vì `j > far`).
3. Cập nhật `far = max(far, i + maxJump)`.
4. Chạm `n − 1` ⇒ `True`.

In [11]:
def can_reach_bfs_far(s, minJump, maxJump):
    n = len(s)
    q = deque([0])
    far = 0
    while q:
        i = q.popleft()
        start = max(i + minJump, far + 1)
        end = min(i + maxJump, n - 1)
        for j in range(start, end + 1):
            if s[j] == '0':
                if j == n - 1:
                    return True
                q.append(j)
        far = max(far, end)
    return n == 1

for (s, lo, hi), want in examples:
    print(can_reach_bfs_far(s, lo, hi), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False


Lưu ý `far = max(far, end)` (dùng `end` đã cắt ở `n − 1`, không phải `i + maxJump`) để `range` không chạy ngoài mảng. Vì `n ≥ 2`, dòng `return n == 1` thực chất trả `False`.

**Độ phức tạp:** mỗi chỉ số được quét (vòng `for`) tối đa **một** lần do `start > far` ⇒ Time `O(n)`, Space `O(n)`.

### 7.4 Cách 5 — Tham lam theo "đoạn tới được" (mảng hiệu)

Góc nhìn "đẩy" nhưng không duyệt từng cạnh: khi `ok[i] = True`, đánh dấu cả đoạn `[i + minJump, i + maxJump]` là "có thể tới" bằng **mảng hiệu** `diff[i + minJump] += 1`, `diff[i + maxJump + 1] −= 1`. Khi quét tới `j`, tổng dồn `run` của `diff` cho biết có bao nhiêu đoạn phủ `j`.

In [12]:
def can_reach_diff(s, minJump, maxJump):
    n = len(s)
    diff = [0] * (n + 1)
    run = 0
    ok_last = False
    for j in range(n):
        run += diff[j]
        ok = (j == 0) or (s[j] == '0' and run > 0)
        if ok:
            a, b = j + minJump, min(j + maxJump, n - 1)
            if a <= b:
                diff[a] += 1
                diff[b + 1] -= 1
        if j == n - 1:
            ok_last = ok
    return ok_last

for (s, lo, hi), want in examples:
    print(can_reach_diff(s, lo, hi), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False


**Độ phức tạp:** Time `O(n)`, Space `O(n)`. Cách này cho thấy mối liên hệ: "kéo + đếm cửa sổ" (Cách 3) và "đẩy + mảng hiệu" (Cách 5) là hai mặt của cùng một ý tưởng.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 `s = "011010"`, `minJump = 2`, `maxJump = 3`:

| j | vào (`j−2`) | ra (`j−4`) | cnt | s[j] | ok[j] |
|---|---|---|---|---|---|
| 1 | — | — | 0 | 1 | F |
| 2 | ok[0]=T | — | 1 | 1 | F |
| 3 | ok[1]=F | — | 1 | 0 | **T** |
| 4 | ok[2]=F | ok[0]=T | 0 | 1 | F |
| 5 | ok[3]=T | ok[1]=F | 1 | 0 | **T** |

In [13]:
def trace_window(s, minJump, maxJump):
    n = len(s); ok = [False] * n; ok[0] = True; cnt = 0
    print(f"j=0: điểm xuất phát, ok[0]=True")
    for j in range(1, n):
        inn = out = "-"
        if j - minJump >= 0:
            cnt += ok[j - minJump]; inn = f"ok[{j - minJump}]={ok[j - minJump]}"
        if j - maxJump - 1 >= 0:
            cnt -= ok[j - maxJump - 1]; out = f"ok[{j - maxJump - 1}]={ok[j - maxJump - 1]}"
        ok[j] = s[j] == '0' and cnt > 0
        print(f"j={j}: vào {inn:12s} ra {out:12s} cnt={cnt} s[j]={s[j]} -> ok[{j}]={ok[j]}")
    return ok[-1]

trace_window("011010", 2, 3)

j=0: điểm xuất phát, ok[0]=True
j=1: vào -            ra -            cnt=0 s[j]=1 -> ok[1]=False
j=2: vào ok[0]=True   ra -            cnt=1 s[j]=1 -> ok[2]=False
j=3: vào ok[1]=False  ra -            cnt=1 s[j]=0 -> ok[3]=True
j=4: vào ok[2]=False  ra ok[0]=True   cnt=0 s[j]=1 -> ok[4]=False
j=5: vào ok[3]=True   ra ok[1]=False  cnt=1 s[j]=0 -> ok[5]=True


True

In [14]:
trace_window("01101110", 2, 3)

j=0: điểm xuất phát, ok[0]=True
j=1: vào -            ra -            cnt=0 s[j]=1 -> ok[1]=False
j=2: vào ok[0]=True   ra -            cnt=1 s[j]=1 -> ok[2]=False
j=3: vào ok[1]=False  ra -            cnt=1 s[j]=0 -> ok[3]=True
j=4: vào ok[2]=False  ra ok[0]=True   cnt=0 s[j]=1 -> ok[4]=False
j=5: vào ok[3]=True   ra ok[1]=False  cnt=1 s[j]=1 -> ok[5]=False
j=6: vào ok[4]=False  ra ok[2]=False  cnt=1 s[j]=1 -> ok[6]=False
j=7: vào ok[5]=False  ra ok[3]=True   cnt=0 s[j]=0 -> ok[7]=False


False

In [15]:
def trace_bfs_far(s, minJump, maxJump):
    n = len(s); q = deque([0]); far = 0
    while q:
        i = q.popleft()
        start, end = max(i + minJump, far + 1), min(i + maxJump, n - 1)
        added = [j for j in range(start, end + 1) if s[j] == '0']
        print(f"lấy i={i}: quét [{start}, {end}] (far cũ={far}) -> thêm {added}")
        if n - 1 in added:
            print("chạm ô cuối -> True"); return True
        q.extend(added); far = max(far, end)
    print("hàng đợi rỗng -> False"); return False

trace_bfs_far("0000100100", 1, 3)

lấy i=0: quét [1, 3] (far cũ=0) -> thêm [1, 2, 3]
lấy i=1: quét [4, 4] (far cũ=3) -> thêm []
lấy i=2: quét [5, 5] (far cũ=4) -> thêm [5]
lấy i=3: quét [6, 6] (far cũ=5) -> thêm [6]
lấy i=5: quét [7, 8] (far cũ=6) -> thêm [8]
lấy i=6: quét [9, 9] (far cũ=8) -> thêm [9]
chạm ô cuối -> True


True

Trong trace BFS, chú ý mỗi lần quét bắt đầu từ `far + 1` — các ô đã quét không bao giờ bị quét lại, dù khoảng nhảy của các ô liên tiếp chồng lên nhau.

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def canReach(self, s: str, minJump: int, maxJump: int) -> bool:
        n = len(s)
        if s[-1] != '0':
            return False
        ok = [False] * n
        ok[0] = True
        cnt = 0
        for j in range(1, n):
            if j >= minJump:
                cnt += ok[j - minJump]
            if j > maxJump:
                cnt -= ok[j - maxJump - 1]
            ok[j] = s[j] == '0' and cnt > 0
        return ok[-1]

sol = Solution()
for (s, lo, hi), want in examples:
    got = sol.canReach(s, lo, hi)
    print(got, "OK" if got == want else "SAI")

True OK
False OK


## 10. Vì sao thuật toán đúng

**DP đúng.** Định nghĩa `ok[j]` = "tồn tại dãy nhảy hợp lệ từ 0 tới `j`". Một dãy nhảy tới `j > 0` có bước cuối từ `i` với `j − i ∈ [minJump, maxJump]`, `s[j] = '0'`, và phần trước là dãy hợp lệ tới `i`. Ngược lại, có `i` như vậy với `ok[i]` thì nối thêm bước `i → j`. Vậy `ok[j] = (s[j] == '0') ∧ ∃ i ∈ W(j) : ok[i]`, với `W(j) = [j − maxJump, j − minJump] ∩ [0, j − 1]`.

**Bộ đếm cửa sổ đúng (invariant).** Sau khi xử lý bước `j`, `cnt = Σ_{i ∈ W(j)} ok[i]`. Chứng minh quy nạp: `W(j)` so với `W(j − 1)` thêm đúng phần tử `j − minJump` (nếu `≥ 0`) và bớt đúng phần tử `j − maxJump − 1` (nếu `≥ 0`). Các `ok` đó đã được tính (chỉ số `< j`). Do đó `cnt > 0 ⇔ ∃ i ∈ W(j): ok[i]`.

**BFS với `far` đúng.** Hai nhận xét:
- Các ô được thêm vào hàng đợi theo thứ tự chỉ số **tăng nghiêm ngặt** (mỗi ô mới có chỉ số `> far`, mà `far ≥` mọi chỉ số đã thêm trước đó). Do đó các ô cũng được **lấy ra** theo thứ tự tăng.
- Giả sử khi lấy `i`, ta bỏ qua một `j ≤ far` với `j ∈ [i + minJump, i + maxJump]`. Chỉ số `j ≤ far` nghĩa là đã có một ô `i' < i` lấy ra trước với `j ≤ i' + maxJump`; vì `i' < i` nên `i' + minJump < i + minJump ≤ j`. Vậy `j ∈ [i' + minJump, i' + maxJump]` — `j` đã được xét khi xử lý `i'` (hoặc trước đó). Không bỏ sót ô nào. ✓

Vì mỗi chỉ số nằm trong đúng một lần quét (các đoạn `[start, end]` rời nhau), tổng công việc `O(n)`.

Mục 12 kiểm chứng chéo trên hàng nghìn input ngẫu nhiên.

## 11. Phân tích Time / Space

Gọi `n = len(s) ≤ 10⁵`, `k = maxJump − minJump + 1`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS ngây thơ | `O(n · k)` | `O(n)` | worst `~n²/2` |
| 1b. DP kéo ngây thơ | `O(n · k)` | `O(n)` | |
| 2. DP + tổng tiền tố | `O(n)` | `O(n)` | hai mảng |
| 3. DP + đếm cửa sổ | `O(n)` | `O(n)` | **nộp bài** |
| 4. BFS + biên `far` | `O(n)` | `O(n)` | mỗi ô quét 1 lần |
| 5. Mảng hiệu | `O(n)` | `O(n)` | góc nhìn "đẩy" |

In [17]:
random.seed(1)
n = 10**5
cases = {
    "toàn 0, ô cuối 1": ("0" * (n - 1) + "1", 1, n - 1),
    "ngẫu nhiên 30% '1'": ("0" + "".join('1' if random.random() < 0.3 else '0' for _ in range(n - 1)), 3, 50),
    "toàn 0, khoảng rộng": ("0" * n, 2, 40000),
}
for name, (s, lo, hi) in cases.items():
    line = f"{name:22s}:"
    res = []
    for fname, fn in [("prefix", can_reach_prefix), ("window", can_reach_window), ("bfs_far", can_reach_bfs_far),
                      ("diff", can_reach_diff), ("submit", Solution().canReach)]:
        t0 = time.perf_counter(); res.append(fn(s, lo, hi)); line += f" {fname}={time.perf_counter() - t0:.3f}s"
    assert len(set(res)) == 1
    print(line, "->", res[0])

toàn 0, ô cuối 1      : prefix=0.028s window=0.000s bfs_far=0.040s diff=0.026s submit=0.000s -> False


ngẫu nhiên 30% '1'    : prefix=0.012s window=0.012s bfs_far=0.015s diff=0.014s submit=0.009s -> True


toàn 0, khoảng rộng   : prefix=0.014s window=0.012s bfs_far=0.014s diff=0.016s submit=0.008s -> True


## 12. Bộ test

Dữ liệu ngẫu nhiên đúng ràng buộc: `s[0] = '0'`, `1 ≤ minJump ≤ maxJump < len(s)`. So mọi cách với BFS ngây thơ (đúng theo định nghĩa).

In [18]:
fns = [can_reach_dp_naive, can_reach_prefix, can_reach_window, can_reach_bfs_far, can_reach_diff,
       lambda s, a, b: Solution().canReach(s, a, b)]

for (s, lo, hi), want in examples:
    assert can_reach_bfs_naive(s, lo, hi) == want
    for fn in fns:
        assert fn(s, lo, hi) == want

edge = [
    (("00", 1, 1), True),
    (("01", 1, 1), False),           # ô cuối là '1'
    (("010", 1, 1), False),          # phải dẫm lên '1'
    (("010", 2, 2), True),           # nhảy qua '1'
    (("0000", 2, 2), False),         # chỉ chạm chỉ số chẵn, n-1 = 3 lẻ
    (("00000", 2, 2), True),
    (("0" * 10, 9, 9), True),        # một bước thẳng tới đích
    (("0110", 1, 2), False),
]
for (s, lo, hi), want in edge:
    assert can_reach_bfs_naive(s, lo, hi) == want, (s, lo, hi)
    for fn in fns:
        assert fn(s, lo, hi) == want, (fn, s, lo, hi)

random.seed(2024)
cnt_true = 0
for _ in range(3000):
    n = random.randint(2, 40)
    p = random.choice([0.1, 0.3, 0.5])
    s = "0" + "".join('1' if random.random() < p else '0' for _ in range(n - 1))
    lo = random.randint(1, n - 1); hi = random.randint(lo, n - 1)
    want = can_reach_bfs_naive(s, lo, hi)
    cnt_true += want
    for fn in fns:
        assert fn(s, lo, hi) == want, (fn, s, lo, hi)
print(f"ví dụ + edge + 3000 bộ ngẫu nhiên khớp BFS ngây thơ ({cnt_true} bộ True)")

ví dụ + edge + 3000 bộ ngẫu nhiên khớp BFS ngây thơ (1278 bộ True)


In [19]:
# n lớn hơn: các cách O(n) khớp DP ngây thơ khi khoảng nhảy hẹp
random.seed(77)
for _ in range(40):
    n = random.randint(500, 3000)
    s = "0" + "".join('1' if random.random() < 0.4 else '0' for _ in range(n - 1))
    lo = random.randint(1, 20); hi = lo + random.randint(0, 30)
    hi = min(hi, n - 1)
    want = can_reach_dp_naive(s, lo, hi)
    assert can_reach_prefix(s, lo, hi) == can_reach_window(s, lo, hi) == can_reach_bfs_far(s, lo, hi) == can_reach_diff(s, lo, hi) == want
print("n tới 3000: các cách O(n) khớp DP ngây thơ")

n tới 3000: các cách O(n) khớp DP ngây thơ


Edge `("0000", 2, 2)`: chỉ nhảy đúng 2 bước ⇒ chỉ tới chỉ số chẵn, mà đích là 3 ⇒ `False`. Edge `("010", 1, 1)`: buộc dẫm ô 1 là `'1'` ⇒ `False`.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam: luôn nhảy xa nhất có thể

In [20]:
def wrong_greedy_far(s, minJump, maxJump):
    n = len(s); i = 0
    while i < n - 1:
        nxt = None
        for j in range(min(i + maxJump, n - 1), i + minJump - 1, -1):
            if s[j] == '0':
                nxt = j; break
        if nxt is None:
            return False
        i = nxt
    return True

random.seed(3)
for _ in range(20000):
    n = random.randint(3, 12)
    s = "0" + "".join(random.choice("01") for _ in range(n - 1))
    lo = random.randint(1, n - 1); hi = random.randint(lo, n - 1)
    if wrong_greedy_far(s, lo, hi) != can_reach_bfs_naive(s, lo, hi):
        print(f"phản ví dụ: s={s!r}, minJump={lo}, maxJump={hi} | tham lam = {wrong_greedy_far(s, lo, hi)}, "
              f"đúng = {can_reach_bfs_naive(s, lo, hi)}")
        break

phản ví dụ: s='01110101010', minJump=4, maxJump=9 | tham lam = False, đúng = True


Nhảy xa nhất có thể "vượt quá" đích trong phạm vi `minJump` hoặc rơi vào vùng chết. **Sửa:** xét tất cả ô tới được (DP/BFS).

### Lỗi 2 — Cửa sổ lệch một: dùng `j − maxJump` làm phần tử ra

Phần tử rời cửa sổ khi chuyển sang `j` là `j − maxJump − 1`, không phải `j − maxJump` (chỉ số `j − maxJump` vẫn còn trong cửa sổ của `j`).

In [21]:
def wrong_window_off(s, minJump, maxJump):
    n = len(s); ok = [False] * n; ok[0] = True; cnt = 0
    for j in range(1, n):
        if j - minJump >= 0:
            cnt += ok[j - minJump]
        if j - maxJump >= 0:
            cnt -= ok[j - maxJump]            # SAI: phải là j - maxJump - 1
        ok[j] = s[j] == '0' and cnt > 0
    return ok[-1]

for (s, lo, hi), want in examples:
    print("sai =", wrong_window_off(s, lo, hi), "| đúng =", want)

sai = False | đúng = True
sai = False | đúng = False


Với `minJump == maxJump`, phần tử vừa vào cũng bị trừ ngay ⇒ `cnt` luôn 0. **Sửa:** `cnt −= ok[j − maxJump − 1]` khi `j − maxJump − 1 ≥ 0`. Mẹo kiểm tra: thử `minJump == maxJump`.

### Lỗi 3 — Chỉ cập nhật bộ đếm khi `s[j] == '0'` (dùng `continue` quá sớm)

Nếu `continue` ngay khi gặp `'1'`, các phần tử "vào/ra" tại bước đó bị bỏ qua ⇒ `cnt` lệch với cửa sổ thật.

In [22]:
def wrong_skip_update(s, minJump, maxJump):
    n = len(s); ok = [False] * n; ok[0] = True; cnt = 0
    for j in range(1, n):
        if s[j] == '1':
            continue                          # SAI: bỏ qua cập nhật cnt
        if j - minJump >= 0:
            cnt += ok[j - minJump]
        if j - maxJump - 1 >= 0:
            cnt -= ok[j - maxJump - 1]
        ok[j] = cnt > 0
    return ok[-1]

random.seed(4)
for _ in range(20000):
    n = random.randint(3, 12)
    s = "0" + "".join(random.choice("01") for _ in range(n - 1))
    lo = random.randint(1, n - 1); hi = random.randint(lo, n - 1)
    if wrong_skip_update(s, lo, hi) != can_reach_bfs_naive(s, lo, hi):
        print(f"phản ví dụ: s={s!r}, minJump={lo}, maxJump={hi} | sai = {wrong_skip_update(s, lo, hi)}, "
              f"đúng = {can_reach_bfs_naive(s, lo, hi)}")
        break

phản ví dụ: s='0101100', minJump=2, maxJump=5 | sai = False, đúng = True


**Sửa:** luôn cập nhật `cnt` ở mọi `j`, chỉ quyết định `ok[j]` theo `s[j]`. (Ở Cách 2 với tổng tiền tố thì dùng `if s[j] == '0'` được vì `pre` vẫn cập nhật bên ngoài khối `if`.)

### Lỗi 4 — BFS dùng `far = i + maxJump` không cắt biên, hoặc quên `seen` ở bản ngây thơ

Quên `seen` ở BFS ngây thơ ⇒ một ô bị đưa vào hàng đợi rất nhiều lần ⇒ bùng nổ thời gian/bộ nhớ.

In [23]:
def bfs_no_seen_count(s, minJump, maxJump, limit=200000):
    n = len(s); q = deque([0]); pushes = 0
    while q and pushes < limit:
        i = q.popleft()
        if i == n - 1:
            return True, pushes
        for j in range(i + minJump, min(i + maxJump, n - 1) + 1):
            if s[j] == '0':
                q.append(j); pushes += 1          # SAI: không đánh dấu đã thăm
    return None, pushes

res, pushes = bfs_no_seen_count("0" * 30 + "1", 1, 5)
print(f"không dùng seen: dừng sau {pushes} lần đẩy (giới hạn để không treo), kết quả = {res}")
print("có seen:", can_reach_bfs_naive("0" * 30 + "1", 1, 5))

không dùng seen: dừng sau 200000 lần đẩy (giới hạn để không treo), kết quả = None
có seen: False


**Sửa:** đánh dấu `seen[j] = True` khi đẩy vào hàng đợi; tốt hơn nữa dùng biên `far` (Cách 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Từ `i` nhảy tới bất kỳ `j` trong `[i + a, i + b]`" ⇒ **đảo chiều thành cửa sổ tiền nhiệm** `[j − b, j − a]` + tổng tiền tố / bộ đếm trượt.
- Câu hỏi "có tồn tại" trên cửa sổ ⇒ đếm (`cnt > 0`); câu hỏi "max/min" trên cửa sổ ⇒ deque đơn điệu (như Jump Game VI).
- BFS trên các khoảng chồng nhau ⇒ giữ **biên đã quét** để mỗi phần tử xét một lần.
- Luôn thử `minJump == maxJump` và `s[-1] == '1'` khi kiểm tra.
- Họ bài: 55/45 Jump Game I/II, 1306 Jump Game III (BFS), 1696 Jump Game VI (deque), 1340 Jump Game V.

## 15. Biến thể của bài

1. **Số bước nhảy ít nhất** tới đích: BFS theo tầng + biên `far` (mỗi ô vào hàng đợi một lần, ghi khoảng cách).
2. **Đếm số cách** tới đích (modulo `10⁹+7`): `ways[j] = Σ_{i ∈ W(j)} ways[i]` — đúng khung tổng tiền tố, đổi bool thành số.
3. **Có thể nhảy lùi** (`j ∈ [i − maxJump, i − minJump]` cũng được): không còn thứ tự topo, cần BFS với cấu trúc tìm "ô chưa thăm tiếp theo" (DSU / sorted set).
4. **Mỗi ô có chi phí, tối thiểu tổng chi phí**: `dp[j] = cost[j] + min dp[W(j)]` ⇒ deque đơn điệu.

In [24]:
MOD = 10**9 + 7

def min_jumps(s, minJump, maxJump):
    """Biến thể 1: số bước ít nhất, -1 nếu không tới được."""
    n = len(s); dist = [-1] * n; dist[0] = 0; q = deque([0]); far = 0
    while q:
        i = q.popleft()
        start, end = max(i + minJump, far + 1), min(i + maxJump, n - 1)
        for j in range(start, end + 1):
            if s[j] == '0':
                dist[j] = dist[i] + 1; q.append(j)
        far = max(far, end)
    return dist[-1]

def count_ways(s, minJump, maxJump):
    """Biến thể 2: số dãy nhảy khác nhau tới đích (mod 1e9+7)."""
    n = len(s); ways = [0] * n; ways[0] = 1; pre = [0] * (n + 1); pre[1] = 1
    for j in range(1, n):
        if s[j] == '0':
            l, r = max(0, j - maxJump), j - minJump
            if r >= 0:
                ways[j] = (pre[r + 1] - pre[l]) % MOD
        pre[j + 1] = (pre[j] + ways[j]) % MOD
    return ways[-1]

def count_ways_brute(s, minJump, maxJump):
    n = len(s)
    def go(i):
        if i == n - 1: return 1
        return sum(go(j) for j in range(i + minJump, min(i + maxJump, n - 1) + 1) if s[j] == '0')
    return go(0)

def min_jumps_brute(s, minJump, maxJump):
    n = len(s); dist = {0: 0}; q = deque([0])
    while q:
        i = q.popleft()
        for j in range(i + minJump, min(i + maxJump, n - 1) + 1):
            if s[j] == '0' and j not in dist:
                dist[j] = dist[i] + 1; q.append(j)
    return dist.get(n - 1, -1)

random.seed(5)
for _ in range(500):
    n = random.randint(2, 14)
    s = "0" + "".join(random.choice("001") for _ in range(n - 1))
    lo = random.randint(1, n - 1); hi = random.randint(lo, n - 1)
    assert min_jumps(s, lo, hi) == min_jumps_brute(s, lo, hi)
    assert count_ways(s, lo, hi) == count_ways_brute(s, lo, hi) % MOD
print("ví dụ 1: ít bước nhất =", min_jumps("011010", 2, 3), "| số cách =", count_ways("011010", 2, 3))

ví dụ 1: ít bước nhất = 2 | số cách = 1


Lưu ý: BFS với biên `far` vẫn cho khoảng cách **ngắn nhất** đúng, vì một ô `j` lần đầu được quét bởi ô `i` có khoảng cách nhỏ nhất trong số các ô có thể tới `j` (các ô lấy ra theo thứ tự khoảng cách không giảm và chỉ số tăng) — đã kiểm chứng bằng `min_jumps_brute` ở trên.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS ngây thơ | thử mọi bước | `O(nk)` | `O(n)` | kiểm chứng |
| 1b. DP kéo ngây thơ | `any` trên cửa sổ | `O(nk)` | `O(n)` | cầu nối tư duy |
| 2. DP + tiền tố | hiệu tổng tiền tố | `O(n)` | `O(n)` | dễ mở rộng sang đếm cách |
| 3. DP + đếm cửa sổ | `cnt` vào/ra | `O(n)` | `O(n)` | **nộp bài** |
| 4. BFS + `far` | quét mỗi ô một lần | `O(n)` | `O(n)` | khi cần số bước ít nhất |
| 5. Mảng hiệu | đẩy đoạn tới được | `O(n)` | `O(n)` | góc nhìn "đẩy" |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `canReach("00111010", 3, 5)` và `canReach("0000000000", 8, 8)`?

**Bài 2 (tính tay):** lập bảng `j, cnt, ok[j]` của Cách 3 cho `s = "0010010"`, `minJump = 2`, `maxJump = 3`.

**Bài 3 (điền TODO):** hoàn thiện `reachable_set(s, minJump, maxJump)` trả về danh sách mọi chỉ số tới được (theo thứ tự tăng).

**Bài 4 (tự viết lại):** viết lại Cách 3 **không dùng mảng `ok` đầy đủ** — gợi ý: chỉ cần nhớ các `ok` trong `maxJump + 1` vị trí gần nhất (deque hoặc mảng vòng).

**Bài 5 (mini challenge):** biến thể **được nhảy cả hai chiều** (`|j − i| ∈ [minJump, maxJump]`). Viết lời giải `O(n log n)` hoặc `O(n α(n))` (gợi ý: DSU "nút chưa thăm tiếp theo" hoặc `bisect` trên danh sách ô `'0'` chưa thăm) và kiểm chứng bằng BFS ngây thơ. (Biến thể tự đặt.)

In [25]:
def reachable_set(s, minJump, maxJump):
    # TODO: chạy Cách 3, trả về [j for j in range(n) if ok[j]]
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `"00111010"`: từ 0 nhảy 3..5 ⇒ chỉ ô 5 là `'0'`. (2) Từ 5 nhảy ≥ 3 ⇒ ≥ 8 vượt mảng; đích là 7. (3) Đáp án `False`; chuỗi 10 số 0 với nhảy đúng 8: `0 → 8`, đích 9 ⇒ `False`.

**Bài 2:** (1) `ok[0] = T`. (2) `j = 2`: cnt = 1, `s[2] = '1'` ⇒ F; `j = 3`: cnt = 1 ⇒ T. (3) Tiếp tục tới `j = 6`: kiểm tra bằng `trace_window`.

**Bài 3:** (1) Copy Cách 3. (2) Giữ mảng `ok`. (3) Trả về chỉ số có `ok[j]`.

**Bài 4:** (1) `ok[j − maxJump − 1]` là phần tử xa nhất cần đọc. (2) Mảng vòng kích thước `maxJump + 1`, chỉ số `j % (maxJump + 1)`. (3) Cẩn thận: ghi `ok[j]` sau khi đã đọc phần tử ra (cùng ô vòng!).

**Bài 5:** (1) Danh sách `free` các ô `'0'` chưa thăm, sắp tăng. (2) Lấy ô `i`, dùng `bisect` tìm các ô trong `[i + lo, i + hi]` và `[i − hi, i − lo]`, thêm vào hàng đợi rồi xóa khỏi `free`. (3) Xóa khỏi list `O(n)` — dùng DSU "next" để đạt gần tuyến tính.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().canReach("00111010", 3, 5) is False
assert Solution().canReach("0000000000", 8, 8) is False

# Bài 2
def ok_table(s, minJump, maxJump):
    n = len(s); ok = [False] * n; ok[0] = True; cnt = 0
    for j in range(1, n):
        if j >= minJump: cnt += ok[j - minJump]
        if j > maxJump: cnt -= ok[j - maxJump - 1]
        ok[j] = s[j] == '0' and cnt > 0
    return ok
print("Bài 2:", ok_table("0010010", 2, 3))

# Bài 3
def reachable_set_ans(s, minJump, maxJump):
    return [j for j, v in enumerate(ok_table(s, minJump, maxJump)) if v]

# Bài 4 — mảng vòng kích thước maxJump + 1
def can_reach_ring(s, minJump, maxJump):
    n = len(s); W = maxJump + 1; ring = [False] * W; ring[0] = True; cnt = 0; last = True
    for j in range(1, n):
        if j >= minJump: cnt += ring[(j - minJump) % W]
        if j > maxJump: cnt -= ring[(j - maxJump - 1) % W]     # cùng ô với j % W: đọc trước khi ghi
        cur = s[j] == '0' and cnt > 0
        ring[j % W] = cur; last = cur
    return last

random.seed(91)
for _ in range(2000):
    n = random.randint(2, 30)
    s = "0" + "".join(random.choice("01") for _ in range(n - 1))
    lo = random.randint(1, n - 1); hi = random.randint(lo, n - 1)
    assert can_reach_ring(s, lo, hi) == can_reach_bfs_naive(s, lo, hi)
print("Bài 1-4: OK")

Bài 2: [True, False, False, True, False, False, True]
Bài 1-4: OK


In [27]:
# kiểm tra Bài 3 kỹ hơn: tập tới được so với BFS ngây thơ (lấy seen)
def reachable_bfs(s, minJump, maxJump):
    n = len(s); seen = [False] * n; seen[0] = True; q = deque([0])
    while q:
        i = q.popleft()
        for j in range(i + minJump, min(i + maxJump, n - 1) + 1):
            if s[j] == '0' and not seen[j]:
                seen[j] = True; q.append(j)
    return [j for j in range(n) if seen[j]]

random.seed(92)
for _ in range(1000):
    n = random.randint(2, 30)
    s = "0" + "".join(random.choice("01") for _ in range(n - 1))
    lo = random.randint(1, n - 1); hi = random.randint(lo, n - 1)
    assert reachable_set_ans(s, lo, hi) == reachable_bfs(s, lo, hi)
print("Bài 3: tập chỉ số tới được khớp BFS")

Bài 3: tập chỉ số tới được khớp BFS


In [28]:
# Bài 5 — nhảy hai chiều, DSU "ô '0' chưa thăm kế tiếp"
def can_reach_both_dirs(s, minJump, maxJump):
    n = len(s)
    nxt = list(range(n + 1))                 # nxt[x] = ô '0' chưa thăm nhỏ nhất >= x (n = lính canh)
    def find(x):
        root = x
        while nxt[root] != root: root = nxt[root]
        while nxt[x] != root: nxt[x], x = root, nxt[x]
        return root
    for i in range(n):
        if s[i] == '1': nxt[i] = i + 1       # ô '1' không bao giờ thăm
    nxt[0] = 1                               # đã thăm ô 0
    q = deque([0])
    while q:
        i = q.popleft()
        if i == n - 1: return True
        for a, b in ((i + minJump, i + maxJump), (i - maxJump, i - minJump)):
            a, b = max(a, 0), min(b, n - 1)
            j = find(a) if a <= b else n
            while j <= b:
                q.append(j); nxt[j] = j + 1; j = find(j + 1)
    return False

def both_dirs_brute(s, minJump, maxJump):
    n = len(s); seen = {0}; q = deque([0])
    while q:
        i = q.popleft()
        for j in list(range(i + minJump, i + maxJump + 1)) + list(range(i - maxJump, i - minJump + 1)):
            if 0 <= j < n and s[j] == '0' and j not in seen:
                seen.add(j); q.append(j)
    return n - 1 in seen

random.seed(93)
diff_cnt = 0
for _ in range(2000):
    n = random.randint(2, 25)
    s = "0" + "".join(random.choice("01") for _ in range(n - 1))
    lo = random.randint(1, n - 1); hi = random.randint(lo, n - 1)
    assert can_reach_both_dirs(s, lo, hi) == both_dirs_brute(s, lo, hi), (s, lo, hi)
    diff_cnt += both_dirs_brute(s, lo, hi) != can_reach_bfs_naive(s, lo, hi)
print(f"Bài 5: khớp BFS hai chiều; nhảy lùi giúp tới đích ở {diff_cnt}/2000 bộ mà một chiều không tới được")

Bài 5: khớp BFS hai chiều; nhảy lùi giúp tới đích ở 86/2000 bộ mà một chiều không tới được


## 20. Template nhớ nhanh + Checklist

```python
# "Nhảy trong khoảng [a, b]" -> kéo từ cửa sổ [j-b, j-a], đếm số ô tới được
ok = [False] * n; ok[0] = True; cnt = 0
for j in range(1, n):
    if j >= a: cnt += ok[j - a]          # phần tử vào cửa sổ
    if j > b:  cnt -= ok[j - b - 1]      # phần tử ra khỏi cửa sổ
    ok[j] = s[j] == '0' and cnt > 0      # cập nhật cnt ở MỌI j
return ok[-1]
```

**Checklist trước khi submit:**
- [ ] Phần tử ra khỏi cửa sổ là `j − maxJump − 1` (thử `minJump == maxJump`).
- [ ] Cập nhật bộ đếm ở mọi `j`, kể cả khi `s[j] == '1'`.
- [ ] `s[-1] == '1'` ⇒ `False`.
- [ ] Không dùng tham lam "nhảy xa nhất".
- [ ] BFS: đánh dấu đã thăm hoặc dùng biên `far` để tránh `O(n²)`.